# Portrait LoRA — Исследование данных и экспериментальный notebook

Запускайте ячейки по порядку.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image

from stage1_data.make_prompts import attrs_to_prompt, CELEBA_ATTRS, parse_attr_file

print('Импорты OK')

## 1. EDA — Анализ атрибутов CelebA

In [ ]:
# Загружаем атрибуты
attrs_map = parse_attr_file(Path('../data/raw/list_attr_celeba.txt'))
print(f'Загружено: {len(attrs_map):,} записей')

# Подсчёт
from collections import Counter
counts = Counter()
for rec in attrs_map.values():
    for attr, val in rec.items():
        if val == 1:
            counts[attr] += 1

n = len(attrs_map)
sorted_attrs = sorted(counts.items(), key=lambda x: -x[1])

In [ ]:
# График распределения атрибутов
fig, ax = plt.subplots(figsize=(14, 6))
attrs_sorted = [x[0] for x in sorted_attrs]
pcts = [counts[a] / n * 100 for a in attrs_sorted]

colors = ['#d9534f' if p < 5 else '#5bc0de' for p in pcts]
bars = ax.barh(attrs_sorted, pcts, color=colors)
ax.axvline(5, color='red', linestyle='--', alpha=0.5, label='Порог 5%')
ax.set_xlabel('% от датасета')
ax.set_title('Распределение атрибутов CelebA (красный = редкий <5%)')
ax.legend()
plt.tight_layout()
plt.savefig('attr_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Сохранено: attr_distribution.png')

## 2. Тестирование шаблонизатора промптов

In [ ]:
# Примеры промптов для разных комбинаций атрибутов
test_cases = [
    # (описание, атрибуты)
    ('Молодая блондинка, улыбается',
     {'Male':0,'Young':1,'Blond_Hair':1,'Wavy_Hair':1,'Smiling':1,'Wearing_Earrings':1}),
    ('Пожилой мужчина с бородой и очками',
     {'Male':1,'Young':0,'Gray_Hair':1,'Goatee':1,'Eyeglasses':1}),
    ('Средний возраст, яркий макияж',
     {'Male':0,'Young':0,'Brown_Hair':1,'Heavy_Makeup':1,'High_Cheekbones':1}),
    ('Молодой мужчина, чёрные волосы',
     {'Male':1,'Young':1,'Black_Hair':1,'Straight_Hair':1,'Smiling':1}),
    ('Лысый, пожилой',
     {'Male':1,'Young':0,'Bald':1,'Eyeglasses':1}),
]

for desc, attrs in test_cases:
    prompt = attrs_to_prompt(attrs)
    print(f'[{desc}]')
    print(f'  → {prompt}')
    print()

## 3. Визуализация изображений датасета

In [ ]:
# Показываем 16 случайных изображений после предобработки
manifest_path = '../data/processed/train_proc.json'

if Path(manifest_path).exists():
    with open(manifest_path) as f:
        records = json.load(f)

    sample = random.sample(records, 16)

    fig, axes = plt.subplots(4, 4, figsize=(12, 12))
    for ax, rec in zip(axes.flat, sample):
        img = Image.open(rec['image_path'])
        ax.imshow(img)
        # Короткая подпись
        attrs = rec['attrs']
        gender = 'M' if attrs.get('Male') else 'F'
        age    = 'Y' if attrs.get('Young') else 'O'
        ax.set_title(f'{gender},{age}', fontsize=8)
        ax.axis('off')

    plt.suptitle('Случайные изображения из train (M=мужчина, F=женщина, Y=молодой, O=пожилой)')
    plt.tight_layout()
    plt.savefig('train_samples.png', dpi=150)
    plt.show()
else:
    print(f'Файл не найден: {manifest_path}')
    print('Запустите предобработку: python stage2_preprocess/preprocess.py')

## 4. Проверка Dataset и DataLoader

In [ ]:
from stage2_preprocess.dataset import CelebADataset

ds = CelebADataset('../data/processed/train_proc.json', max_samples=100)
print(f'Размер датасета: {len(ds)}')

sample = ds[0]
print(f'pixel_values: {sample["pixel_values"].shape}',
      f'∈ [{sample["pixel_values"].min():.2f}, {sample["pixel_values"].max():.2f}]')
print(f'input_ids:    {sample["input_ids"].shape}')
print(f'prompt:       {sample["prompt"]}')

# Визуализация нескольких пар (изображение + промпт)
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, ax in enumerate(axes):
    item = ds[i]
    # Денормализация [-1,1] → [0,1]
    img = (item['pixel_values'] * 0.5 + 0.5).permute(1,2,0).numpy()
    img = np.clip(img, 0, 1)
    ax.imshow(img)
    ax.set_title(item['prompt'][:50] + '...', fontsize=6, wrap=True)
    ax.axis('off')
plt.tight_layout()
plt.savefig('dataset_check.png', dpi=150)
plt.show()
print('✓ Dataset работает')

## 5. Просмотр результатов генерации

In [ ]:
# Сравнение: SD без LoRA vs SD с LoRA
samples_dir = Path('../samples')

step_files = sorted(samples_dir.glob('step_*.png'))
if step_files:
    print(f'Найдено {len(step_files)} файлов с примерами генерации')
    # Показываем последние 3 чекпоинта
    for f in step_files[-3:]:
        print(f'  {f.name}')
        img = Image.open(f)
        plt.figure(figsize=(14, 4))
        plt.imshow(img)
        plt.title(f.stem)
        plt.axis('off')
        plt.show()
else:
    print('Нет файлов samples/step_*.png')
    print('Запустите обучение: python stage3_train/train_lora.py')

## 6. Таблица метрик

In [ ]:
import json

results_file = Path('../evaluation_results.json')
if results_file.exists():
    results = json.load(open(results_file))
    targets = {
        'fid':        ('≤20',   results.get('fid', -1) <= 20),
        'lpips':      ('≤0.28', results.get('lpips', 1) <= 0.28),
        'ssim':       ('≥0.72', results.get('ssim', 0) >= 0.72),
        'clip_score': ('≥0.28', results.get('clip_score', 0) >= 0.28),
    }
    print('Метрика    Результат   Цель    Статус')
    print('-' * 45)
    for m, (tgt, ok) in targets.items():
        val = results.get(m, '—')
        status = '✓' if ok else '✗'
        print(f'{m:12s} {val:8.4f}    {tgt:6s}  {status}')
else:
    print('evaluation_results.json не найден')
    print('Запустите: python stage4_eval/metrics.py ...')